# Preparação da base SISU 2023 — mulheres
## Consolida os 4 arquivos do SISU 2023 em uma base única pronta para modelagem

---

### Arquivos necessários (baixar em dados.gov.br → SISU)

| Arquivo | Descrição | Link |
|---------|-----------|------|
| `chamada_regular_sisu_2023_1.csv` | Chamada Regular 1/2023 | dados.gov.br |
| `chamada_regular_sisu_2023_2.csv` | Chamada Regular 2/2023 | dados.gov.br |
| `lista_espera_sisu_2023_1.csv` | Lista de Espera 1/2023 | dados.gov.br |
| `lista_espera_sisu_2023_2.csv` | Lista de Espera 2/2023 | dados.gov.br |

### Por que usar os 4 arquivos?

O SISU 2023 teve **2 edições** (1/2023 e 2/2023), cada uma com Chamada Regular e Lista de Espera. São populações distintas de candidatos, mas para o **Modelo 2** (escolha de curso tech) o que importa é a **candidatura declarada** — não se o candidato foi chamado ou não.

Por isso unimos todos os 4 e deduplicamos por candidata × curso.

### Saída gerada
- `sisu_2023_mulheres.csv` → base consolidada, opção 1, já tratada, pronta para o Modelo 2
- `sisu_2023_mulheres_opcao2.csv` → opção 2 (para EDA comparativa)

## Célula 0 — Montar Drive e definir caminhos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PASTA = '/content/drive/MyDrive/TCC/'   # ← ajuste se necessário
SAIDA = PASTA + 'sisu_2023_mulheres.csv'

Mounted at /content/drive


## Célula 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import os
print("Pronto.")

Pronto.


## Célula 2 — Definição dos arquivos e colunas

Carregamos apenas as colunas necessárias para o modelo — reduz uso de memória em ~60%.

In [ ]:
ARQUIVOS_SISU = [
    {'path': PASTA + 'chamada_regular_sisu_2023_1.csv',
     'tipo': 'Chamada Regular', 'edicao': 1, 'desc': 'Chamada Regular 1/2023'},
    {'path': PASTA + 'chamada_regular_sisu_2023_2.csv',
     'tipo': 'Chamada Regular', 'edicao': 2, 'desc': 'Chamada Regular 2/2023'},
    {'path': PASTA + 'lista_espera_sisu_2023_1.csv',
     'tipo': 'Lista de Espera', 'edicao': 1, 'desc': 'Lista de Espera 1/2023'},
    {'path': PASTA + 'lista_espera_sisu_2023_2.csv',
     'tipo': 'Lista de Espera', 'edicao': 2, 'desc': 'Lista de Espera 2/2023'},
]

COLS_USAR = [
    'ANO', 'EDICAO', 'ETAPA', 'DS_ETAPA',
    'NOME_CURSO', 'GRAU', 'TURNO',
    'UF_IES', 'MUNICIPIO_CAMPUS',
    'MOD_CONCORRENCIA', 'TP_COTA', 'TIPO_MOD_CONCORRENCIA',
    'SEXO', 'DT_NASCIMENTO', 'UF_CANDIDATO', 'MUNICIPIO_CANDIDATO',
    'NOTA_L', 'NOTA_CH', 'NOTA_CN', 'NOTA_M', 'NOTA_R',
    'NOTA_CANDIDATO', 'NOTA_CORTE',
    'OPCAO', 'APROVADO', 'MATRICULA', 'CLASSIFICACAO',
]
print(f"{len(COLS_USAR)} colunas selecionadas")

27 colunas selecionadas


## Célula 3 — Leitura e consolidação dos 4 arquivos

In [ ]:
dfs = []

for arq in ARQUIVOS_SISU:
    path = arq['path']

    if not os.path.exists(path):
        print(f"  ⚠️  NÃO ENCONTRADO: {path}")
        continue

    print(f"Lendo {arq['desc']}...", end=' ')

    # Detectar colunas disponíveis (lista de espera pode ter menos colunas)
    cols_disp = pd.read_csv(path, sep='|', encoding='latin1', nrows=0).columns.tolist()
    cols_ler  = [c for c in COLS_USAR if c in cols_disp]

    df_arq = pd.read_csv(path, sep='|', encoding='latin1',
                         low_memory=False, usecols=cols_ler)
    df_arq['TIPO_ARQUIVO'] = arq['tipo']
    dfs.append(df_arq)
    print(f"{len(df_arq):,} linhas")

if not dfs:
    raise FileNotFoundError("Nenhum arquivo encontrado. Verifique os caminhos.")

df_bruto = pd.concat(dfs, ignore_index=True)
print(f"\nTotal consolidado: {len(df_bruto):,} linhas")
print(f"Arquivos carregados: {len(dfs)} de {len(ARQUIVOS_SISU)}")

Lendo Chamada Regular 1/2023... 2,062,815 linhas
Lendo Chamada Regular 2/2023... 578,781 linhas
  ⚠️  NÃO ENCONTRADO: /content/drive/MyDrive/TCC/lista_espera_sisu_2023_1.csv
  ⚠️  NÃO ENCONTRADO: /content/drive/MyDrive/TCC/lista_espera_sisu_2023_2.csv

Total consolidado: 2,641,596 linhas
Arquivos carregados: 2 de 4


## Célula 4 — Filtro: apenas mulheres

In [ ]:
df_f = df_bruto[df_bruto['SEXO'] == 'F'].copy()
print(f"Mulheres: {len(df_f):,} ({len(df_f)/len(df_bruto)*100:.1f}% do total)")
print(f"Homens:   {(df_bruto['SEXO']=='M').sum():,}")

print("\nPor tipo de arquivo:")
print(df_f['TIPO_ARQUIVO'].value_counts().to_string())
print("\nPor edição:")
if 'EDICAO' in df_f.columns:
    print(df_f['EDICAO'].value_counts().to_string())

Mulheres: 1,669,928 (63.2% do total)
Homens:   971,668

Por tipo de arquivo:
TIPO_ARQUIVO
Chamada Regular    1669928

Por edição:
EDICAO
1    1283223
2     386705


## Célula 5 — Deduplicação

### Por que deduplica?
Uma mesma candidata aparece múltiplas vezes porque:
- Escolheu 2 opções de curso (`OPCAO` = 1 e 2)
- Aparece em edições diferentes (1/2023 e 2/2023)
- Pode aparecer em Chamada Regular **e** Lista de Espera

### Estratégia
- Separamos **opção 1** (preferência real) de **opção 2**
- Para o mesmo par candidata × curso em edições diferentes, mantemos a linha com **melhor resultado** (aprovada > pendente)

In [ ]:
# Prioridade de resultado para desempate
PRIO_MAT = {'EFETIVADA':4,'DEFERIDA':3,'PENDENTE':2,
            'DOCUMENTACAO REJEITADA':1,'CANCELADA':1,'NÃO COMPARECEU':0}
PRIO_APR = {'S':2, 'N':1}

def deduplicar(df_opcao):
    df = df_opcao.copy()
    if 'MATRICULA' in df.columns:
        df['_prio_mat'] = df['MATRICULA'].map(PRIO_MAT).fillna(0)
    else:
        df['_prio_mat'] = 0
    if 'APROVADO' in df.columns:
        df['_prio_apr'] = df['APROVADO'].map(PRIO_APR).fillna(0)
    else:
        df['_prio_apr'] = 0

    # Ordenar: melhor resultado primeiro
    df = df.sort_values(['_prio_apr','_prio_mat'], ascending=False)

    # Chave: candidata × curso × IES (sem CPF, que está mascarado)
    CHAVE = [c for c in ['SEXO','DT_NASCIMENTO','UF_CANDIDATO',
                          'MUNICIPIO_CANDIDATO','NOME_CURSO','UF_IES']
             if c in df.columns]

    df_dedup = df.drop_duplicates(subset=CHAVE, keep='first')
    df_dedup = df_dedup.drop(columns=['_prio_mat','_prio_apr'])
    return df_dedup

df_op1 = deduplicar(df_f[df_f['OPCAO'] == 1])
df_op2 = deduplicar(df_f[df_f['OPCAO'] == 2])

print(f"Opção 1 após dedup: {len(df_op1):,} candidatas únicas")
print(f"Opção 2 após dedup: {len(df_op2):,} candidatas únicas")

Opção 1 após dedup: 385,929 candidatas únicas
Opção 2 após dedup: 393,829 candidatas únicas


## Célula 6 — Tratamento das variáveis

Aplica todos os tratamentos de uma vez:
- Notas: vírgula → ponto decimal
- Idade derivada de `DT_NASCIMENTO`
- Região geográfica da candidata e da IES
- Flag de mobilidade (candidata em região diferente da IES)
- Modalidade simplificada (Ampla / Cota)
- **Target `IS_TECH`** — curso de tecnologia sim/não

In [ ]:
REGIOES = {
    'AC':'N','AM':'N','AP':'N','PA':'N','RO':'N','RR':'N','TO':'N',
    'AL':'NE','BA':'NE','CE':'NE','MA':'NE','PB':'NE','PE':'NE',
    'PI':'NE','RN':'NE','SE':'NE',
    'DF':'CO','GO':'CO','MS':'CO','MT':'CO',
    'ES':'SE','MG':'SE','RJ':'SE','SP':'SE',
    'PR':'S','RS':'S','SC':'S'
}

CURSOS_TECH = {
    'CIÊNCIA DA COMPUTAÇÃO',
    'CIÊNCIAS DA COMPUTAÇÃO',
    'ABI - CIÊNCIA DA COMPUTAÇÃO',
    'SISTEMAS DE INFORMAÇÃO',
    'SISTEMA DE INFORMAÇÃO',
    'ANÁLISE E DESENVOLVIMENTO DE SISTEMAS',
    'ENGENHARIA DE COMPUTAÇÃO',
    'ENGENHARIA DA COMPUTAÇÃO',
    'ENGENHARIA COMPUTACIONAL',
    'ENGENHARIA DE COMPUTAÇÃO E INFORMAÇÃO',
    'ENGENHARIA ELETRÔNICA E DE COMPUTAÇÃO',
    'ENGENHARIA DE SOFTWARE',
    'ENGENHARIA DE SISTEMAS',
    'REDES DE COMPUTADORES',
    'SISTEMAS DE TELECOMUNICAÇÕES',
    'SISTEMAS PARA INTERNET',
    'SISTEMAS E MÍDIAS DIGITAIS',
    'COMPUTAÇÃO',
    'COMPUTAÇÃO E INFORMÁTICA',
    'JOGOS DIGITAIS',
    'DESIGN DE JOGOS',
    'SEGURANÇA DA INFORMAÇÃO',
    'GESTÃO DE DADOS',
    'GESTÃO DA TECNOLOGIA DA INFORMAÇÃO',
    'TECNOLOGIA DA INFORMAÇÃO',
    'TECNOLOGIAS DA INFORMAÇÃO E COMUNICAÇÃO',
    'TELEMÁTICA',
    'GEOPROCESSAMENTO',
    'INFORMÁTICA',
    'INTELIGÊNCIA ARTIFICIAL',
    'CIÊNCIA DE DADOS',
    'CIÊNCIA DE DADOS E INTELIGÊNCIA ARTIFICIAL',
    'CIÊNCIA DE DADOS PARA NEGÓCIOS',
    'CIÊNCIA DOS DADOS',
    'MATEMÁTICA COMPUTACIONAL',
    'MATEMÁTICA APLICADA E COMPUTACIONAL',
    'INTERDISCIPLINAR EM TECNOLOGIA DA INFORMAÇÃO - BI/LI',
    'INTERDISCIPLINAR EM MATEMÁTICA E COMPUTAÇÃO E SUAS TECNOLOGIAS'
}

def tratar(df):
    df = df.copy()

    # Notas: vírgula → ponto
    NOTAS = ['NOTA_L','NOTA_CH','NOTA_CN','NOTA_M','NOTA_R',
             'NOTA_CANDIDATO','NOTA_CORTE']
    for c in NOTAS:
        if c in df.columns:
            df[c] = pd.to_numeric(
                df[c].astype(str).str.replace(',','.',regex=False),
                errors='coerce'
            )

    # Idade
    if 'DT_NASCIMENTO' in df.columns:
        df['DT_NASCIMENTO'] = pd.to_numeric(df['DT_NASCIMENTO'], errors='coerce')
        df['IDADE'] = (2023 - df['DT_NASCIMENTO']).clip(15, 75)

    # Regiões
    if 'UF_CANDIDATO' in df.columns:
        df['REGIAO_CANDIDATA'] = df['UF_CANDIDATO'].map(REGIOES).fillna('N/D')
    if 'UF_IES' in df.columns:
        df['REGIAO_IES'] = df['UF_IES'].map(REGIOES).fillna('N/D')

    # Mobilidade
    if 'REGIAO_CANDIDATA' in df.columns and 'REGIAO_IES' in df.columns:
        df['MOBILIDADE'] = (df['REGIAO_CANDIDATA'] != df['REGIAO_IES']).astype(int)

    # Cota
    if 'MOD_CONCORRENCIA' in df.columns:
        df['COTA'] = df['MOD_CONCORRENCIA'].apply(
            lambda x: 'Ampla' if str(x).lower()=='ampla concorrência' else 'Cota'
        )

    # Target IS_TECH
    if 'NOME_CURSO' in df.columns:
        df['IS_TECH'] = df['NOME_CURSO'].isin(CURSOS_TECH).astype(int)

    return df

df_op1_final = tratar(df_op1)
df_op2_final = tratar(df_op2)
print("Tratamento concluído.")

Tratamento concluído.


## Célula 7 — Diagnóstico da base final

In [ ]:
print("=" * 60)
print("DIAGNÓSTICO — sisu_2023_mulheres (opção 1)")
print("=" * 60)
print(f"Total de candidatas: {len(df_op1_final):,}")

if 'IS_TECH' in df_op1_final.columns:
    n_tech = df_op1_final['IS_TECH'].sum()
    n_tot  = len(df_op1_final)
    print(f"\nTARGET IS_TECH:")
    print(f"  Tech:     {n_tech:,} ({n_tech/n_tot*100:.1f}%)")
    print(f"  Não-tech: {n_tot-n_tech:,} ({(n_tot-n_tech)/n_tot*100:.1f}%)")
    print(f"  Razão desbalanceamento: {(n_tot-n_tech)/n_tech:.0f}:1")

print("\nNOTAS — estatísticas:")
notas = [c for c in ['NOTA_M','NOTA_CN','NOTA_L','NOTA_CH','NOTA_R','NOTA_CANDIDATO']
         if c in df_op1_final.columns]
print(df_op1_final[notas].describe().round(1).to_string())

print("\nREGIÃO DA CANDIDATA:")
if 'REGIAO_CANDIDATA' in df_op1_final.columns:
    print(df_op1_final['REGIAO_CANDIDATA'].value_counts().to_string())

print("\nCOTA:")
if 'COTA' in df_op1_final.columns:
    print(df_op1_final['COTA'].value_counts().to_string())

print("\nTURNO:")
if 'TURNO' in df_op1_final.columns:
    print(df_op1_final['TURNO'].value_counts().to_string())

print("\nTIPO DE ARQUIVO (fonte):")
if 'TIPO_ARQUIVO' in df_op1_final.columns:
    print(df_op1_final['TIPO_ARQUIVO'].value_counts().to_string())

print("\nNULOS POR COLUNA:")
nulos = df_op1_final.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
for col, n in nulos.items():
    print(f"  {col:30s} {n:>7,} ({n/len(df_op1_final)*100:.1f}%)")

DIAGNÓSTICO — sisu_2023_mulheres (opção 1)
Total de candidatas: 385,929

TARGET IS_TECH:
  Tech:     13,789 (3.6%)
  Não-tech: 372,140 (96.4%)
  Razão desbalanceamento: 27:1

NOTAS — estatísticas:
         NOTA_M   NOTA_CN    NOTA_L   NOTA_CH    NOTA_R  NOTA_CANDIDATO
count  385929.0  385929.0  385929.0  385929.0  385929.0        385929.0
mean      550.7     503.5     536.4     545.9     710.9           576.6
std       120.5      79.7      71.2      75.1     156.9            85.0
min         0.0       0.0       0.0       0.0      40.0            84.0
25%       460.0     449.2     492.3     497.8     600.0           515.6
50%       545.0     497.2     542.7     548.0     720.0           572.1
75%       632.1     554.9     586.9     597.4     840.0           635.0
max       983.2     847.2     797.0     839.2    1000.0           964.2

REGIÃO DA CANDIDATA:
REGIAO_CANDIDATA
NE    157468
SE    124905
S      42276
N      35595
CO     25685

COTA:
COTA
Cota     211011
Ampla    174918

TURNO:

## Célula 8 — Salvar as bases

In [ ]:
# Base principal — opção 1 (usada no Modelo 2)
df_op1_final.to_csv(SAIDA, index=False, encoding='utf-8-sig')
print(f"✓ Salvo: {SAIDA}")
print(f"  {len(df_op1_final):,} linhas × {len(df_op1_final.columns)} colunas")

# Base opção 2 — EDA comparativa
saida_op2 = PASTA + 'sisu_2023_mulheres_opcao2.csv'
df_op2_final.to_csv(saida_op2, index=False, encoding='utf-8-sig')
print(f"✓ Salvo: {saida_op2}")
print(f"  {len(df_op2_final):,} linhas × {len(df_op2_final.columns)} colunas")

print("\nCOLUNAS DA BASE FINAL:")
for col in df_op1_final.columns:
    print(f"  {col}")

✓ Salvo: /content/drive/MyDrive/TCC/sisu_2023_mulheres.csv
  385,929 linhas × 34 colunas
✓ Salvo: /content/drive/MyDrive/TCC/sisu_2023_mulheres_opcao2.csv
  393,829 linhas × 34 colunas

COLUNAS DA BASE FINAL:
  ANO
  EDICAO
  ETAPA
  DS_ETAPA
  UF_IES
  MUNICIPIO_CAMPUS
  NOME_CURSO
  GRAU
  TURNO
  TP_COTA
  TIPO_MOD_CONCORRENCIA
  MOD_CONCORRENCIA
  SEXO
  DT_NASCIMENTO
  UF_CANDIDATO
  MUNICIPIO_CANDIDATO
  OPCAO
  NOTA_L
  NOTA_CH
  NOTA_CN
  NOTA_M
  NOTA_R
  NOTA_CANDIDATO
  NOTA_CORTE
  CLASSIFICACAO
  APROVADO
  MATRICULA
  TIPO_ARQUIVO
  IDADE
  REGIAO_CANDIDATA
  REGIAO_IES
  MOBILIDADE
  COTA
  IS_TECH


## Célula 9 — Como usar no notebook principal

Após rodar este script, **substitua** a seção de leitura do SISU no notebook principal pelo código abaixo:

In [ ]:
# ── Substituir no notebook principal ──────────────────────────────────────
# Ao invés de ler e tratar o SISU do zero, carregue a base já preparada:

df_sisu_f = pd.read_csv(
    PASTA + 'sisu_2023_mulheres.csv',
    encoding='utf-8-sig',
    low_memory=False
)

print(f"Base SISU 2023 (mulheres) carregada: {len(df_sisu_f):,} candidatas")
print(f"\nColunas disponíveis:")
print(list(df_sisu_f.columns))

# Conferência rápida do target
print(f"\nDistribuição IS_TECH:")
print(df_sisu_f['IS_TECH'].value_counts())

# ── O que já está tratado nesta base ──────────────────────────────────────
# ✓ Apenas mulheres (SEXO == 'F')
# ✓ Apenas opção 1 (preferência real de curso)
# ✓ Deduplicada por candidata × curso (sem duplicatas de edição/arquivo)
# ✓ Notas convertidas de vírgula para ponto decimal
# ✓ IDADE derivada de DT_NASCIMENTO
# ✓ REGIAO_CANDIDATA e REGIAO_IES
# ✓ MOBILIDADE (1 = candidata e IES em regiões diferentes)
# ✓ COTA ('Ampla' ou 'Cota')
# ✓ IS_TECH — target do Modelo 2 (0=não-tech / 1=tech)
# ✓ TIPO_ARQUIVO — fonte da linha (Chamada Regular / Lista de Espera)

Base SISU 2023 (mulheres) carregada: 385,929 candidatas

Colunas disponíveis:
['ANO', 'EDICAO', 'ETAPA', 'DS_ETAPA', 'UF_IES', 'MUNICIPIO_CAMPUS', 'NOME_CURSO', 'GRAU', 'TURNO', 'TP_COTA', 'TIPO_MOD_CONCORRENCIA', 'MOD_CONCORRENCIA', 'SEXO', 'DT_NASCIMENTO', 'UF_CANDIDATO', 'MUNICIPIO_CANDIDATO', 'OPCAO', 'NOTA_L', 'NOTA_CH', 'NOTA_CN', 'NOTA_M', 'NOTA_R', 'NOTA_CANDIDATO', 'NOTA_CORTE', 'CLASSIFICACAO', 'APROVADO', 'MATRICULA', 'TIPO_ARQUIVO', 'IDADE', 'REGIAO_CANDIDATA', 'REGIAO_IES', 'MOBILIDADE', 'COTA', 'IS_TECH']

Distribuição IS_TECH:
IS_TECH
0    373054
1     12875
Name: count, dtype: int64
